# 02b Per-Ticker LSTM Strict Forecasting

Use this notebook when you want to train only the per-ticker LSTM. It writes LSTM artifacts into an isolated `lstm_only/strict_protocol` artifact tree. Run `03_model_comparison.ipynb` afterward to compare these LSTM results with table-model and global-LSTM results.


In [1]:
from pathlib import Path
import os
import sys

cwd = Path.cwd().resolve()
for candidate in [cwd, cwd / "forecasting", cwd.parent, cwd.parent / "forecasting"]:
    if (candidate / "src" / "stock_forecast").exists():
        PROJECT_DIR = candidate
        break
else:
    raise RuntimeError("Cannot locate forecasting project directory with src/stock_forecast")

SRC_DIR = PROJECT_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

ARTIFACT_DIR = PROJECT_DIR / "artifacts"
DATA_DIR = ARTIFACT_DIR / "data"
REPORTS_DIR = ARTIFACT_DIR / "reports"
for path in [DATA_DIR, REPORTS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print(f"PROJECT_DIR = {PROJECT_DIR}")

PROJECT_DIR = /home/sapce/forecasting_stock_prices/forecasting


In [2]:
import importlib.util

import pandas as pd
from IPython.display import display

from stock_forecast.artifacts import load_json, load_table
from stock_forecast.mlflow_tracking import MLflowRunConfig, log_strict_protocol_result, mlflow_horizon_run
from stock_forecast.models import build_model
from stock_forecast.strict_protocol import run_strict_per_ticker_protocol

pd.set_option("display.max_columns", 180)


## Constants

In [3]:
if importlib.util.find_spec("torch") is None:
    raise ImportError("PyTorch is required for this notebook. Install the forecasting[deep] extra.")

FORCE_RETRAIN = True  # set True when you want to retune/retrain LSTM even if matching cache exists
PRIMARY_METRIC = "directional_accuracy"
RANDOM_STATE = 42

STRICT_VALIDATION_ROWS = 126
STRICT_TEST_ROWS = 126
MATURE_MIN_ROWS = 1008
LIMITED_HISTORY_MIN_BLOCK_ROWS = 42
MIN_TRAIN_ROWS = 60
STRICT_MAX_TRAIN_ROWS = 1260
INNER_MAX_FOLDS = 3
INNER_MIN_TRAIN_ROWS = 126
LIMITED_HISTORY_N_TRIALS = 20

LSTM_N_TRIALS = 60
LSTM_OPTUNA_N_JOBS = 1
LSTM_MAX_EPOCHS = 150
LSTM_PATIENCE = 15
LSTM_DEVICE = "auto"
LSTM_ENSEMBLE_SEEDS = [1, 7, 21, 42, 101]

TRANSACTION_COST_BPS = 10
SLIPPAGE_BPS = 5
LONG_THRESHOLD = 0.0
SIGNAL_ANCHOR = "expanding_median"

MLFLOW_ENABLED = os.environ.get("MLFLOW_ENABLED", "true").strip().lower() in {"1", "true", "yes", "y"}
MLFLOW_TRACKING_URI = os.environ.get("MLFLOW_TRACKING_URI", "http://localhost:5000")
MLFLOW_EXPERIMENT_NAME = os.environ.get("MLFLOW_EXPERIMENT_NAME", "stock_return_forecasting_research")
MLFLOW_LOG_OPTUNA_TRIALS = os.environ.get("MLFLOW_LOG_OPTUNA_TRIALS", "true").strip().lower() in {"1", "true", "yes", "y"}

LSTM_ONLY_ARTIFACT_NAME = "lstm_only"

HORIZONS = [
    {"name": "week", "horizon": 5},
    {"name": "month", "horizon": 21},
]


In [4]:
def make_lstm_search_space(horizon: int, limited_history: bool = False) -> dict:
    huber_beta_choices = [0.04, 0.08, 0.12] if horizon >= 21 else [0.02, 0.04, 0.06]
    return {
        "lookback": {"type": "categorical", "choices": [20, 40, 60] if limited_history else [20, 40, 60, 90, 126]},
        "hidden_size": {"type": "categorical", "choices": [16, 32, 64, 96, 128]},
        "num_layers": {"type": "categorical", "choices": [1] if limited_history else [1, 2]},
        "input_projection_size": {"type": "categorical", "choices": [0, 32, 64, 128]},
        "lstm_dropout": {"type": "float", "low": 0.0, "high": 0.35},
        "head_dropout": {"type": "float", "low": 0.15, "high": 0.55},
        "learning_rate": {"type": "float", "low": 1e-4, "high": 2e-3, "log": True},
        "weight_decay": {"type": "float", "low": 1e-5, "high": 1e-2, "log": True},
        "batch_size": {"type": "categorical", "choices": [16, 32, 64] if limited_history else [32, 64, 128]},
        "loss": {"type": "categorical", "choices": ["smooth_l1", "huber", "mse"]},
        "huber_beta": {"type": "categorical", "choices": huber_beta_choices},
        "feature_clip": {"type": "categorical", "choices": [3.0, 5.0, 8.0]},
        "grad_clip_norm": {"type": "float", "low": 0.5, "high": 2.0},
    }


def make_lstm_config(lstm_feature_cols: list[str], horizon: int) -> dict:
    return {
        "name": "lstm",
        "model_type": "lstm",
        "estimator_factory": build_model,
        "input_mode": "full_frame",
        "feature_cols": lstm_feature_cols,
        "static_params": {
            "max_epochs": LSTM_MAX_EPOCHS,
            "patience": LSTM_PATIENCE,
            "device": LSTM_DEVICE,
        },
        "search_space": make_lstm_search_space(horizon, limited_history=False),
        "limited_history_search_space": make_lstm_search_space(horizon, limited_history=True),
        "post_selection_static_params": {"ensemble_seeds": LSTM_ENSEMBLE_SEEDS},
        "n_trials": LSTM_N_TRIALS,
        "optuna_n_jobs": LSTM_OPTUNA_N_JOBS,
        "limited_history_n_trials": LIMITED_HISTORY_N_TRIALS,
        "needs_scaler": False,
    }

## Load LSTM Data Artifacts

In [5]:
horizon_inputs = []

for spec in HORIZONS:
    horizon_name = spec["name"]
    horizon = int(spec["horizon"])
    base_feature_payload = load_json(DATA_DIR / "horizons" / horizon_name / "feature_columns.json")
    target_col = base_feature_payload["target_column"]

    lstm_dir = DATA_DIR / "lstm" / "horizons" / horizon_name
    lstm_model_path = lstm_dir / "model_dataset.parquet"
    lstm_feature_path = lstm_dir / "feature_columns.json"
    if not lstm_model_path.exists() and not lstm_model_path.with_suffix(".csv").exists():
        raise FileNotFoundError(
            f"Missing LSTM data for {horizon_name}: {lstm_model_path}. "
            "Run notebooks/01b_lstm_eda.ipynb first."
        )
    if not lstm_feature_path.exists():
        raise FileNotFoundError(f"Missing LSTM feature payload for {horizon_name}: {lstm_feature_path}")

    lstm_payload = load_json(lstm_feature_path)
    if lstm_payload["target_column"] != target_col:
        raise ValueError(f"LSTM target mismatch for {horizon_name}: {lstm_payload['target_column']} != {target_col}")

    model_df = load_table(lstm_model_path)
    model_df["date"] = pd.to_datetime(model_df["date"])
    lstm_feature_cols = lstm_payload["feature_columns"]
    model_configs = [make_lstm_config(lstm_feature_cols, horizon)]

    horizon_inputs.append({
        "horizon_name": horizon_name,
        "horizon": horizon,
        "model_df": model_df,
        "feature_cols": base_feature_payload["feature_columns"],
        "lstm_feature_cols": lstm_feature_cols,
        "target_col": target_col,
        "model_configs": model_configs,
        "lstm_only_artifact_dir": ARTIFACT_DIR / "horizons" / horizon_name / LSTM_ONLY_ARTIFACT_NAME,
    })

    print({
        "horizon": horizon_name,
        "rows": len(model_df),
        "lstm_features": len(lstm_feature_cols),
        "target": target_col,
        "lstm_only_artifact_dir": str((ARTIFACT_DIR / "horizons" / horizon_name / LSTM_ONLY_ARTIFACT_NAME).relative_to(PROJECT_DIR)),
    })
    display(model_df[["date", "ticker", target_col, *lstm_feature_cols[:6]]].head())


{'horizon': 'week', 'rows': 19407, 'lstm_features': 160, 'target': 'target_return_5_next_open', 'lstm_only_artifact_dir': 'artifacts/horizons/week/lstm_only'}


,date,ticker,target_return_5_next_open,log_close,ret_1,open_close_ret,high_low_range,close_to_high,close_to_low
0,2015-11-09,CBOM,0.013316,1.321756,-0.003992,0.000000,0.000000,0.000000,0.000000
1,2015-11-10,CBOM,0.019908,1.320422,-0.001334,0.004013,0.004005,0.000000,0.004021
2,2015-11-11,CBOM,0.005312,1.319086,-0.001336,0.002677,0.002674,0.000000,0.002681
3,2015-11-12,CBOM,0.010582,1.323088,0.004003,0.000000,0.000000,0.000000,0.000000
4,2015-11-13,CBOM,-0.002649,1.329724,0.006636,0.005305,0.300265,-0.227783,0.005319


{'horizon': 'month', 'rows': 19295, 'lstm_features': 160, 'target': 'target_return_21_next_open', 'lstm_only_artifact_dir': 'artifacts/horizons/month/lstm_only'}


,date,ticker,target_return_21_next_open,log_close,ret_1,open_close_ret,high_low_range,close_to_high,close_to_low
0,2015-11-09,CBOM,0.027761,1.321756,-0.003992,0.000000,0.000000,0.000000,0.000000
1,2015-11-10,CBOM,0.023842,1.320422,-0.001334,0.004013,0.004005,0.000000,0.004021
2,2015-11-11,CBOM,0.021081,1.319086,-0.001336,0.002677,0.002674,0.000000,0.002681
3,2015-11-12,CBOM,0.021053,1.323088,0.004003,0.000000,0.000000,0.000000,0.000000
4,2015-11-13,CBOM,0.011834,1.329724,0.006636,0.005305,0.300265,-0.227783,0.005319


## Train LSTM Only

In [6]:
lstm_results = {}

for item in horizon_inputs:
    horizon_name = item["horizon_name"]
    horizon = item["horizon"]
    mlflow_config = MLflowRunConfig(
        tracking_uri=MLFLOW_TRACKING_URI,
        experiment_name=MLFLOW_EXPERIMENT_NAME,
        notebook_name="02b_lstm_forecasting",
        horizon_name=horizon_name,
        horizon=horizon,
        enabled=MLFLOW_ENABLED,
        log_optuna_trials=MLFLOW_LOG_OPTUNA_TRIALS,
    )
    mlflow_params = {
        "primary_metric": PRIMARY_METRIC,
        "force_retrain": FORCE_RETRAIN,
        "random_state": RANDOM_STATE,
        "lstm_n_trials": LSTM_N_TRIALS,
        "lstm_optuna_n_jobs": LSTM_OPTUNA_N_JOBS,
        "lstm_max_epochs": LSTM_MAX_EPOCHS,
        "lstm_patience": LSTM_PATIENCE,
        "lstm_ensemble_seeds": LSTM_ENSEMBLE_SEEDS,
        "feature_count": len(item["lstm_feature_cols"]),
        "target_col": item["target_col"],
        "validation_rows": STRICT_VALIDATION_ROWS,
        "test_rows": STRICT_TEST_ROWS,
        "transaction_cost_bps": TRANSACTION_COST_BPS,
        "slippage_bps": SLIPPAGE_BPS,
        "signal_anchor": SIGNAL_ANCHOR,
    }
    with mlflow_horizon_run(
        mlflow_config,
        params=mlflow_params,
        tags={"training_protocol": "strict_per_ticker_lstm", "training_notebook": "02b_lstm_forecasting"},
    ) as mlflow_run:
        result = run_strict_per_ticker_protocol(
            model_df=item["model_df"],
            feature_cols=item["feature_cols"],
            target_col=item["target_col"],
            model_configs=item["model_configs"],
            artifact_dir=item["lstm_only_artifact_dir"],
            force_retrain=FORCE_RETRAIN,
            primary_metric=PRIMARY_METRIC,
            random_state=RANDOM_STATE,
            run_metadata={
                "horizon_name": horizon_name,
                "horizon": horizon,
                "training_notebook": "02b_lstm_forecasting",
                "lstm_only_run": True,
            },
            validation_rows=STRICT_VALIDATION_ROWS,
            test_rows=STRICT_TEST_ROWS,
            mature_min_rows=MATURE_MIN_ROWS,
            limited_history_min_block_rows=LIMITED_HISTORY_MIN_BLOCK_ROWS,
            min_train_rows=MIN_TRAIN_ROWS,
            max_train_rows=STRICT_MAX_TRAIN_ROWS,
            inner_max_folds=INNER_MAX_FOLDS,
            inner_min_train_rows=INNER_MIN_TRAIN_ROWS,
            limited_history_n_trials=LIMITED_HISTORY_N_TRIALS,
            transaction_cost_bps=TRANSACTION_COST_BPS,
            slippage_bps=SLIPPAGE_BPS,
            long_threshold=LONG_THRESHOLD,
            signal_anchor=SIGNAL_ANCHOR,
            mlflow_trial_logger=mlflow_run.trial_logger,
        )
        log_strict_protocol_result(
            result,
            params=mlflow_params,
            tags={"training_protocol": "strict_per_ticker_lstm", "training_notebook": "02b_lstm_forecasting"},
        )
    lstm_results[horizon_name] = result

    print(f"=== LSTM-only strict protocol: {horizon_name} ({horizon} trading days) ===")
    display(result["validation_model_ranking"])
    display(result["test_prediction_metrics"])
    display(result["test_signal_metrics"])
    display(result["leakage_audit"])

    failed = result["leakage_audit"][~result["leakage_audit"]["passed"]]
    if not failed.empty:
        raise AssertionError(f"LSTM-only leakage audit failed for {horizon_name}: {failed['check'].tolist()}")


🏃 View run week-lstm-trial-0 at: http://localhost:5000/#/experiments/2/runs/03ed658f96194ee1b2e6d5b35c11ccc9
🧪 View experiment at: http://localhost:5000/#/experiments/2
🏃 View run week-lstm-trial-1 at: http://localhost:5000/#/experiments/2/runs/37c082008e3b44fc9d96c94e600d616f
🧪 View experiment at: http://localhost:5000/#/experiments/2
🏃 View run week-lstm-trial-2 at: http://localhost:5000/#/experiments/2/runs/9410e6f8a39a450aa61291ddb6bb21ba
🧪 View experiment at: http://localhost:5000/#/experiments/2
🏃 View run week-lstm-trial-3 at: http://localhost:5000/#/experiments/2/runs/1339f015fbba4605bfc4ecab1e937756
🧪 View experiment at: http://localhost:5000/#/experiments/2
🏃 View run week-lstm-trial-4 at: http://localhost:5000/#/experiments/2/runs/f4520b6c14de4a22aeba2ed27687a3c9
🧪 View experiment at: http://localhost:5000/#/experiments/2
🏃 View run week-lstm-trial-5 at: http://localhost:5000/#/experiments/2/runs/27d735a50ba34c608a9f40e24c105086
🧪 View experiment at: http://localhost:5000/#/

,split_role,ticker,model_name,n_obs,mae,rmse,r2,pearson,spearman,validation_directional_accuracy,horizon_name,horizon,split_quality,limited_history,n_train,n_train_before_purge,n_train_purged,n_train_available,max_train_rows,train_start,train_end,validation_rank,is_validation_selected
0,validation,CBOM,lstm,126,0.040798,0.053580,-0.019573,0.127722,0.160861,0.539683,week,5,mature,False,1254,1260,6,2506,1260,2020-10-19,2025-08-27,1,True
1,validation,MBNK,lstm,107,0.031755,0.039951,-0.778303,0.049785,0.040224,0.467290,week,5,limited_history,True,317,323,6,323,1260,2024-09-05,2025-10-06,1,True
2,validation,SBER,lstm,126,0.018986,0.024399,-0.130739,0.109235,0.093501,0.492063,week,5,mature,False,1254,1260,6,4377,1260,2020-10-19,2025-08-27,1,True
3,validation,SBERP,lstm,126,0.017720,0.023213,-0.070505,0.032317,0.001749,0.460317,week,5,mature,False,1254,1260,6,4377,1260,2020-10-19,2025-08-27,1,True
4,validation,SVCB,lstm,126,0.030408,0.039251,-0.225646,-0.091873,-0.117846,0.547619,week,5,limited_history,True,378,384,6,384,1260,2024-04-27,2025-08-27,1,True
5,validation,T,lstm,126,0.024632,0.032218,-0.035112,-0.021386,-0.086086,0.468254,week,5,mature,False,1254,1260,6,1381,1260,2020-09-07,2025-08-19,1,True
6,validation,VTBR,lstm,126,0.022967,0.028678,-0.117038,-0.020879,0.051612,0.492063,week,5,mature,False,1254,1260,6,4333,1260,2020-10-13,2025-08-27,1,True


,split_role,ticker,model_name,n_obs,mae,rmse,r2,pearson,spearman,directional_accuracy,horizon_name,horizon,split_quality,limited_history,n_train,n_train_before_purge,n_train_purged,n_refit_available,max_train_rows,train_start,train_end,validation_rank,is_validation_selected,validation_directional_accuracy
0,test,CBOM,lstm,126,0.049306,0.092517,-0.046358,-0.134147,-0.135856,0.388889,week,5,mature,False,1254,1260,6,2632,1260,2021-04-20,2026-01-18,1,True,0.539683
1,test,MBNK,lstm,107,0.013236,0.016505,-0.072765,0.152652,0.130146,0.644860,week,5,limited_history,True,424,430,6,430,1260,2024-09-05,2026-02-06,1,True,0.467290
2,test,SBER,lstm,126,0.010453,0.012837,-0.476896,0.289453,0.265836,0.571429,week,5,mature,False,1254,1260,6,4503,1260,2021-04-20,2026-01-18,1,True,0.492063
3,test,SBERP,lstm,126,0.007950,0.009783,0.081467,0.319072,0.301957,0.571429,week,5,mature,False,1254,1260,6,4503,1260,2021-04-20,2026-01-18,1,True,0.460317
4,test,SVCB,lstm,126,0.024117,0.031398,-0.686307,-0.041226,0.024735,0.571429,week,5,limited_history,True,504,510,6,510,1260,2024-04-27,2026-01-18,1,True,0.547619
5,test,T,lstm,126,0.015213,0.019959,-0.006354,0.131045,0.181648,0.595238,week,5,mature,False,1254,1260,6,1507,1260,2021-03-09,2026-01-12,1,True,0.468254
6,test,VTBR,lstm,126,0.030230,0.041466,-0.071279,-0.109071,-0.168924,0.468254,week,5,mature,False,1254,1260,6,4459,1260,2021-04-14,2026-01-18,1,True,0.492063


,cumulative_return,annualized_return,annualized_volatility,periods_per_year,sharpe,sortino,max_drawdown,calmar,turnover,number_of_trades,ticker,model_name,signal_mode,n_rebalances,sample_warning
0,-0.150949,-0.279112,0.060210,252.0,-4.635654,-4.630435,-0.161643,-1.726712,0.036508,23,CBOM,lstm,overlapping_tranches,126,False
1,-0.095360,-0.210242,0.046923,252.0,-4.480557,-5.530781,-0.117536,-1.788745,0.052336,28,MBNK,lstm,overlapping_tranches,107,False
2,0.040769,0.083199,0.022798,252.0,3.649421,5.818833,-0.010386,8.010645,0.034921,22,SBER,lstm,overlapping_tranches,126,False
3,0.046662,0.095501,0.020093,252.0,4.753003,13.557603,-0.005332,17.910154,0.036508,23,SBERP,lstm,overlapping_tranches,126,False
4,-0.051624,-0.100583,0.029740,252.0,-3.382037,-2.506247,-0.069910,-1.438744,0.015873,10,SVCB,lstm,overlapping_tranches,126,False
5,-0.012485,-0.024815,0.036265,252.0,-0.684260,-0.847128,-0.047910,-0.517954,0.014286,9,T,lstm,overlapping_tranches,126,False
6,-0.013680,-0.027173,0.081764,252.0,-0.332338,-0.451132,-0.125093,-0.217226,0.020635,13,VTBR,lstm,overlapping_tranches,126,False
7,-0.152279,-0.274027,0.173587,50.4,-1.578615,-1.721590,-0.191957,-1.427541,0.269231,7,CBOM,lstm,non_overlapping,26,False
8,-0.114536,-0.243214,0.104851,50.4,-2.319625,-3.169364,-0.152538,-1.594448,0.227273,5,MBNK,lstm,non_overlapping,22,False
9,0.057656,0.114784,0.050989,50.4,2.251165,13.893202,-0.004551,25.221640,0.384615,10,SBER,lstm,non_overlapping,26,False


,check,passed,details
0,strict outer splits are available,True,split_rows=7
1,validation predictions are available,True,rows=863
2,test predictions are available,True,rows=863
3,outer split dates are chronological,True,bad_rows=0
4,train and refit windows respect max_train_rows,True,"train_over_cap=0, refit_over_cap=0"
5,validation predictions match outer split dates,True,"out_of_window=0, wrong_role=0"
6,test predictions match outer split dates,True,"out_of_window=0, wrong_role=0"
7,final refit target dates end before test starts,True,overlap_rows=0
8,final model payloads exist for test predictions,True,missing_models=0


🏃 View run month-lstm-trial-0 at: http://localhost:5000/#/experiments/2/runs/e5d461150143416288a5cab213e7152e
🧪 View experiment at: http://localhost:5000/#/experiments/2
🏃 View run month-lstm-trial-1 at: http://localhost:5000/#/experiments/2/runs/2bbdcff97e2b4311b841199d67961387
🧪 View experiment at: http://localhost:5000/#/experiments/2
🏃 View run month-lstm-trial-2 at: http://localhost:5000/#/experiments/2/runs/87bdd5e7562b44edbfbcb08d0e10a112
🧪 View experiment at: http://localhost:5000/#/experiments/2
🏃 View run month-lstm-trial-3 at: http://localhost:5000/#/experiments/2/runs/1dc65f5b59c041349e2fbc7cc3aed616
🧪 View experiment at: http://localhost:5000/#/experiments/2
🏃 View run month-lstm-trial-4 at: http://localhost:5000/#/experiments/2/runs/4a6e8866b60949878f72ffb2c12fee76
🧪 View experiment at: http://localhost:5000/#/experiments/2
🏃 View run month-lstm-trial-5 at: http://localhost:5000/#/experiments/2/runs/27856929ee29427eb49d01086fffdc39
🧪 View experiment at: http://localhost:5

,split_role,ticker,model_name,n_obs,mae,rmse,r2,pearson,spearman,validation_directional_accuracy,horizon_name,horizon,split_quality,limited_history,n_train,n_train_before_purge,n_train_purged,n_train_available,max_train_rows,train_start,train_end,validation_rank,is_validation_selected
0,validation,CBOM,lstm,126,0.092275,0.108165,0.047765,0.439552,0.476493,0.658730,month,21,mature,False,1238,1260,22,2490,1260,2020-09-25,2025-07-22,1,True
1,validation,MBNK,lstm,104,0.038649,0.045623,-0.861250,0.018551,-0.038835,0.778846,month,21,limited_history,True,291,313,22,313,1260,2024-09-05,2025-09-08,1,True
2,validation,SBER,lstm,126,0.046689,0.058747,-1.635128,-0.119352,-0.043708,0.460317,month,21,mature,False,1238,1260,22,4361,1260,2020-09-25,2025-07-22,1,True
3,validation,SBERP,lstm,126,0.045689,0.056901,-1.790630,-0.470200,-0.403813,0.380952,month,21,mature,False,1238,1260,22,4361,1260,2020-09-25,2025-07-22,1,True
4,validation,SVCB,lstm,124,0.065022,0.083662,-0.498071,-0.259292,-0.285759,0.354839,month,21,limited_history,True,350,372,22,372,1260,2024-04-27,2025-07-26,1,True
5,validation,T,lstm,126,0.046046,0.055880,-0.157750,-0.190266,-0.178685,0.452381,month,21,mature,False,1238,1260,22,1365,1260,2020-08-14,2025-07-14,1,True
6,validation,VTBR,lstm,126,0.063580,0.072411,-1.233909,-0.281288,-0.236541,0.492063,month,21,mature,False,1238,1260,22,4317,1260,2020-09-21,2025-07-22,1,True


,split_role,ticker,model_name,n_obs,mae,rmse,r2,pearson,spearman,directional_accuracy,horizon_name,horizon,split_quality,limited_history,n_train,n_train_before_purge,n_train_purged,n_refit_available,max_train_rows,train_start,train_end,validation_rank,is_validation_selected,validation_directional_accuracy
0,test,CBOM,lstm,126,0.167949,0.182458,-0.823342,-0.096922,-0.090676,0.365079,month,21,mature,False,1238,1260,22,2616,1260,2021-03-29,2025-12-09,1,True,0.658730
1,test,MBNK,lstm,104,0.045377,0.051883,-2.123606,-0.382999,-0.304225,0.211538,month,21,limited_history,True,395,417,22,417,1260,2024-09-05,2026-01-05,1,True,0.778846
2,test,SBER,lstm,126,0.015344,0.019473,-0.734067,0.587379,0.542773,0.698413,month,21,mature,False,1238,1260,22,4487,1260,2021-03-29,2025-12-09,1,True,0.460317
3,test,SBERP,lstm,126,0.016853,0.019394,-0.832970,0.372887,0.397183,0.738095,month,21,mature,False,1238,1260,22,4487,1260,2021-03-29,2025-12-09,1,True,0.380952
4,test,SVCB,lstm,124,0.040215,0.052134,-0.008997,0.144790,0.096636,0.669355,month,21,limited_history,True,474,496,22,496,1260,2024-04-27,2025-12-11,1,True,0.354839
5,test,T,lstm,126,0.039992,0.045812,-0.687692,-0.565901,-0.543727,0.396825,month,21,mature,False,1238,1260,22,1491,1260,2021-02-12,2025-12-01,1,True,0.452381
6,test,VTBR,lstm,126,0.090983,0.113733,-0.319198,-0.575229,-0.596382,0.357143,month,21,mature,False,1238,1260,22,4443,1260,2021-03-23,2025-12-09,1,True,0.492063


,cumulative_return,annualized_return,annualized_volatility,periods_per_year,sharpe,sortino,max_drawdown,calmar,turnover,number_of_trades,ticker,model_name,signal_mode,n_rebalances,sample_warning
0,0.087485,0.182623,0.082664,252.0,2.209220,6.167579,-0.131442,1.389386,0.000756,2,CBOM,lstm,overlapping_tranches,126,False
1,-0.035372,-0.083563,0.013676,252.0,-6.110315,-4.237457,-0.035372,-2.362397,0.004579,10,MBNK,lstm,overlapping_tranches,104,False
2,0.056362,0.115900,0.010590,252.0,10.944537,86.053387,-0.000628,184.440845,0.002268,6,SBER,lstm,overlapping_tranches,126,False
3,0.027840,0.056456,0.007893,252.0,7.152237,258.267234,-0.000201,281.394064,0.006047,16,SBERP,lstm,overlapping_tranches,126,False
4,-0.034055,-0.067993,0.015273,252.0,-4.451794,-2.855042,-0.036519,-1.861825,0.003072,8,SVCB,lstm,overlapping_tranches,124,False
5,-0.075821,-0.145892,0.016243,252.0,-8.981792,-9.048413,-0.076924,-1.896586,0.006803,18,T,lstm,overlapping_tranches,126,False
6,-0.058384,-0.113360,0.060793,252.0,-1.864680,-2.141747,-0.151714,-0.747193,0.003401,9,VTBR,lstm,overlapping_tranches,126,False
7,0.058381,0.120171,0.340925,12.0,0.352485,0.841361,-0.134556,0.893097,0.333333,2,CBOM,lstm,non_overlapping,6,True
8,0.000000,0.000000,0.000000,12.0,NaN,NaN,0.000000,NaN,0.000000,0,MBNK,lstm,non_overlapping,5,True
9,0.026047,0.052773,0.033127,12.0,1.593030,NaN,-0.001499,35.208427,0.666667,4,SBER,lstm,non_overlapping,6,True


,check,passed,details
0,strict outer splits are available,True,split_rows=7
1,validation predictions are available,True,rows=858
2,test predictions are available,True,rows=858
3,outer split dates are chronological,True,bad_rows=0
4,train and refit windows respect max_train_rows,True,"train_over_cap=0, refit_over_cap=0"
5,validation predictions match outer split dates,True,"out_of_window=0, wrong_role=0"
6,test predictions match outer split dates,True,"out_of_window=0, wrong_role=0"
7,final refit target dates end before test starts,True,overlap_rows=0
8,final model payloads exist for test predictions,True,missing_models=0


## Next Step

Run `03_model_comparison.ipynb` after this notebook. The comparison notebook loads `lstm_only` reports directly and combines them with table-model and optional global-LSTM reports.
